# Pharmakaksha · BP701T · Unit IV A · Estimation and t-Tests
**Learn · Grow · Achieve**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajreddys/learn-pharmakaksha-site/blob/main/content/BP701T%20Biostatistics%20and%20Research%20Methodology%20%28Theory%29/BP701T_Unit4A_Estimation_and_t_Tests.ipynb) &nbsp; [Open on learn.pharmakaksha.com](https://learn.pharmakaksha.com/notebooks/index.html?path=BP701T%20Biostatistics%20and%20Research%20Methodology%20%28Theory%29/BP701T_Unit4A_Estimation_and_t_Tests.ipynb)

Companion notebook for BP701T Unit IV A study notes, slides and video. Every worked example from the unit is here, in order, followed by lab exercises and solutions.

> **How to use this notebook:** BP701T is a hand-calculation course. Solve each example with your calculator first, using the study notes, then run the cell to check your answer. In the exam you will not have Python.

## Using learn.pharmakaksha.com
- **First run is slow:** Python loads in your browser, which takes about 10 seconds the first time.
- **Kernel indicator:** the circle at the top right is filled while a cell runs and empty when it is idle.
- **Restart:** Kernel → Restart Kernel if something gets stuck, then run the cells again from the top.
- **Saving:** your changes are saved only in this browser. Download the notebook (File → Download) to keep a copy.
- **Start over:** delete your copy in the file browser to get a fresh notebook from the site.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
print("Ready")

## 4.1 Point and interval estimates
Assay (mg) of 10 paracetamol tablets labelled 500 mg. Hand answers: x̄ = 498.5, s = 3.028, SE = 0.957, 95% CI 496.33 to 500.67.

In [ ]:
asy = np.array([498, 503, 495, 501, 497, 502, 494, 499, 496, 500])
n = len(asy); xbar = asy.mean(); s = asy.std(ddof=1); se = s / np.sqrt(n)
tcrit = stats.t.ppf(0.975, n - 1)
print(f"mean {xbar} | s {s:.3f} | SE {se:.3f} | t(0.05, {n-1}) {tcrit:.3f}")
print(f"95% CI: {xbar - tcrit*se:.2f} to {xbar + tcrit*se:.2f} mg")
print("Same with SciPy:", [round(v, 2) for v in stats.t.interval(0.95, n-1, loc=xbar, scale=se)])

## 4.3 Sample size and power
To detect a 5 mmHg difference (σ = 10) with α = 0.05 (two-sided) and 80% power: n = 2(Z_α/2 + Z_β)²σ²/Δ² ≈ 63 per group.

In [ ]:
za, zb = stats.norm.ppf(0.975), stats.norm.ppf(0.80)
n_formula = 2 * (za + zb)**2 * 10**2 / 5**2
print(f"Z_a/2 = {za:.3f}, Z_b = {zb:.3f}, n = {n_formula:.1f} -> {int(np.ceil(n_formula))} per group")
from statsmodels.stats.power import TTestIndPower
print("t-based answer:", round(TTestIndPower().solve_power(effect_size=0.5, alpha=0.05, power=0.8), 1))
ns = np.arange(10, 151, 10)
plt.plot(ns, [TTestIndPower().power(0.5, n, 0.05) for n in ns], "o-", color="#1F8A70")
plt.axhline(0.8, ls="--", color="#E07A2E"); plt.xlabel("patients per group"); plt.ylabel("power"); plt.title("Power rises with sample size (Δ/σ = 0.5)"); plt.show()

## 4.4 Large-sample Z test
100 patients, mean glucose 108 mg/dL, target 100, σ = 20. Hand answer: Z = 4.0 > 1.96.

In [ ]:
z = (108 - 100) / (20 / np.sqrt(100))
print("Z =", z, "| two-sided p =", round(2 * stats.norm.sf(abs(z)), 6))

## 4.5 One-sample t-test
H₀: μ = 500 mg. Hand answer: t = −1.567, |t| < 2.262 → do not reject H₀.

In [ ]:
res = stats.ttest_1samp(asy, 500)
print(f"t = {res.statistic:.3f}, p = {res.pvalue:.3f}")

## F-test and independent (unpaired) t-test
% dissolved at 30 min: Formulation A vs B (8 tablets each). Hand answers: F = 1.19 (equal variances OK), s²ₚ = 6.5625, t = 4.587 > 2.145.

In [ ]:
A = np.array([82, 85, 88, 80, 84, 86, 83, 87]); B = np.array([78, 81, 76, 80, 79, 77, 82, 75])
F = A.var(ddof=1) / B.var(ddof=1)
print(f"var A {A.var(ddof=1):.3f}, var B {B.var(ddof=1):.3f}, F = {F:.3f}, F table (0.05; 7, 7) = {stats.f.ppf(0.95, 7, 7):.3f}")
sp2 = ((len(A)-1)*A.var(ddof=1) + (len(B)-1)*B.var(ddof=1)) / (len(A) + len(B) - 2)
t = (A.mean() - B.mean()) / np.sqrt(sp2 * (1/len(A) + 1/len(B)))
print(f"pooled variance {sp2:.4f} | t = {t:.3f} | t crit (df 14) = {stats.t.ppf(0.975, 14):.3f}")
print(stats.ttest_ind(A, B))

## Paired t-test
Systolic BP before and after 4 weeks of a new antihypertensive in 8 patients. Hand answers: d̄ = 8.75, s_d = 3.576, t = 6.92 > 2.365.

In [ ]:
before = np.array([152, 160, 148, 165, 158, 170, 155, 162]); after = np.array([144, 150, 146, 152, 150, 158, 149, 151])
d = before - after
print("d =", d, "| mean", d.mean(), "| s_d", round(d.std(ddof=1), 3))
print(stats.ttest_rel(before, after))
print("Wrong analysis (unpaired) for comparison:", stats.ttest_ind(before, after))

## 4.6 Testing correlation and regression
For the Unit III dose–response data, test H₀: ρ = 0 and H₀: slope = 0. In simple regression both give the same t.

In [ ]:
x = np.array([5, 10, 15, 20, 25, 30]); y = np.array([8, 12, 15, 21, 22, 28])
r = stats.pearsonr(x, y).statistic
print("t for r     =", round(r*np.sqrt(4)/np.sqrt(1-r*r), 3))
l = stats.linregress(x, y)
print("t for slope =", round(l.slope / l.stderr, 3), "| p =", round(l.pvalue, 5))

## Worked example · Planning and analysing a BP study
Notes, worked pharma example. Hand answers: t = 6.92; 95% CI 5.76 to 11.74 mmHg; paired sample size 5.4 (Z formula) / 7.6 (t-based) → 8 patients.

In [ ]:
from statsmodels.stats.power import TTestPower
d = before - after; se = d.std(ddof=1) / np.sqrt(len(d))
print("t =", round(d.mean() / se, 2))
print("95% CI:", [round(v, 2) for v in stats.t.interval(0.95, len(d)-1, loc=d.mean(), scale=se)])
z = stats.norm.ppf(0.975) + stats.norm.ppf(0.90)
print("Z-formula n =", round(z**2 * d.std(ddof=1)**2 / 5**2, 1))
print("t-based n   =", round(TTestPower().solve_power(effect_size=5/d.std(ddof=1), alpha=0.05, power=0.9), 1))

## Lab exercises
1. Weights (g) of 8 capsules: 0.51, 0.49, 0.52, 0.50, 0.48, 0.53, 0.50, 0.51. Find the 95% CI and test μ = 0.50 g.
2. Hb (g/dL) before and after iron therapy in 6 patients: (9.8, 11.2), (10.1, 11.0), (8.9, 10.4), (9.5, 10.1), (10.4, 11.9), (9.0, 10.2). Is the rise significant?
3. How many patients per group are needed to detect a 10 mg/dL difference in glucose (σ = 20) with 90% power?
4. n = 64 tablets have mean hardness 5.4 kg/cm²; standard 5.0, σ = 1.2. Is the mean different?

### Solutions

In [ ]:
c = np.array([0.51, 0.49, 0.52, 0.50, 0.48, 0.53, 0.50, 0.51])
print("1:", [round(v, 4) for v in stats.t.interval(0.95, 7, loc=c.mean(), scale=stats.sem(c))], stats.ttest_1samp(c, 0.50))
b4 = np.array([9.8, 10.1, 8.9, 9.5, 10.4, 9.0]); af = np.array([11.2, 11.0, 10.4, 10.1, 11.9, 10.2])
print("2:", stats.ttest_rel(af, b4))
print("3:", np.ceil(2*(stats.norm.ppf(.975) + stats.norm.ppf(.9))**2 * 20**2 / 10**2))
print("4: Z =", round((5.4 - 5.0) / (1.2 / 8), 2))